In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from pathlib import Path

TIMEPOINTS = ["1", "2","3","4",'5']
MAD_THRESHOLD = 3
COMPLICATIONS = ["HDP", "FGR", "sPTB"]
ALL_COMPLICATIONS = ["FGR", "HDP", "sPTB", "PE", "HELLP", "gHTN", "cHTN"]
PROPORTION_THRESHOLD = 0.3
SUPER_CANDIDATE_THRESHOLD = 3

SUBCONDITION_MAPPING = {
    'FGR <5': ['FGR'],
    'Control': ['Control'],
    'gHTN': ['HDP', 'gHTN'],
    'sptb': ['sPTB'],
    'LO severe PE': ['HDP', 'PE'],
    'pp gHTN': ['HDP', 'gHTN'],
    'FGR <5+EO mild PE': ['FGR', 'HDP', 'PE'],
    'pp HELLP': ['HDP', 'HELLP'],
    'FGR <3': ['FGR'],
    'LO severe SIPE': ['HDP', 'PE'],
    'LO FGR <5': ['FGR'],
    'EO severe PE': ['HDP', 'PE'],
    'LO mild PE': ['HDP', 'PE'],
    'sPTB': ['sPTB'],
    'EO severe SIPE (AEDF)': ['HDP', 'PE'],
    'FGR <3+cHTN': ['FGR', 'HDP', 'cHTN'],
    'EO FGR <5 + EO severe PE': ['FGR', 'HDP', 'PE'],
    'FGR <3 + LO severe PE': ['FGR', 'HDP', 'PE'],
    'EO mild SIPE': ['HDP', 'PE'],
    'FGR <5+sptb': ['FGR', 'sPTB'],
    'FGR <5+cHTN': ['FGR', 'HDP', 'cHTN'],
    'LO FGR <3': ['FGR'],
    'EO severe SIPE': ['HDP', 'PE'],
    'LO mild SIPE': ['HDP', 'PE'],
    'sptb+gHTN': ['sPTB', 'HDP','gHTN']
}

In [28]:
modality = "LIPD"
elevated = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/{modality}/biomarker_summary_super_candidates_plasma_elevated.csv")
decreased = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/{modality}/biomarker_summary_super_candidates_plasma_decreased.csv")

In [29]:
allPlasma = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/LIPD/plasma/LIPD_plasma_cleaned_with_metadata.csv")
meta_cols = ["SampleID", "SubjectID", "Batch", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint", "SampleTimepoint",
             "MetadataCanonicalID", "labor_onset", "indicated_onset", "spont_labor_flag", "indicated_onset_flag", "cat_labor_onset_flag",
             "within_0_1wk_delivery_flag", "post_birth_sample_flag"]
dir_output = f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/trajectory_analysis/{modality}"
allPlasma = allPlasma.drop(['MetadataCanonicalID', 'labor_onset',
       'indicated_onset', 'spont_labor_flag', 'indicated_onset_flag',
       'cat_labor_onset_flag', 'within_0_1wk_delivery_flag',
       'post_birth_sample_flag'], axis=1).dropna()

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_87823/361002801.py:1: DtypeWarning: Columns (0: SampleTimepoint) have mixed types. Specify dtype option on import or set low_memory=False.
  allPlasma = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/LIPD/plasma/LIPD_plasma_cleaned_with_metadata.csv")


Gemini explanation for gee without assuming distribution:

"To update this function to use a Generalized Estimating Equation (GEE), we need to replace seaborn's regplot (which assumes independent observations and uses bootstrapping) with the statsmodels GEE implementation.

Since the distribution of your data is unknown, the standard statistical approach with GEE for continuous data is to use the Gaussian family combined with robust (sandwich) standard errors. The robust standard errors ensure that your confidence intervals and p-values remain asymptotically valid even if the assumed distribution or correlation structure is misspecified."

In [30]:
def trajectory_plot_complication_parameters(data, mode, complication):

    #### setting column with category labels and colors for plotting, checking if specified complication exists
    if mode == "pooled_complications": 
        data[mode] = np.where(data["Group"] == "Control", "Control", "Complication")
        colors = {"Control": "gray", "Complication": "red"}

    elif complication == "":
        colors = {"Control": "gray", "HDP": "indianred", "FGR": "goldenrod", "sPTB": "steelblue"}

    elif complication in COMPLICATIONS:
        colors = {"Control": "gray", complication: "red"}

    elif complication in ALL_COMPLICATIONS: # is a more specific condition, mutual exclusivity doesn't apply
        if mode == "mutually_exclusive":
            print(f"Error: non-mutual exclusivity cannot be applied to {complication} specific trajectory plots.")
            return
        colors = {"Control": "gray", complication: "red"}
    else:
        print(f"Error: invalid complication {complication} - cannot generate trajectory plots")
        return

    return data, colors

    

def generate_trajectory_plots(analytes, data, dir_output, tissue, modality, 
                              mode = ["pooled_complications", "mutually_exclusive", "non_mutually_exclusive"], 
                              complication = "", 
                              conf=95):
    
    dir_path = Path(dir_output+"/shapiro-wilk-test")
    dir_path.mkdir(parents=True, exist_ok=True)

    data, colors = trajectory_plot_complication_parameters(data, mode, complication) # get column with category labels and the color dict for plotting


    # Calculate z-score for the given confidence interval
    alpha = 1 - (conf / 100)
    z_score = stats.norm.ppf(1 - alpha / 2)

    analyte_exp = data.copy() #.dropna().copy()

    for m in analytes:
        # Create a new figure for each analyte
        plt.figure(figsize=(8, 6))
        
        # We assume 'meta' was defined earlier in your script, e.g., ["SubjectID", "Group", "SampleGestAge"]
        # Make sure to drop NaNs for the current analyte to prevent GEE from failing
        
        normality = {}
        
        for group in colors.keys():

            # Filter data for the specific group
            if mode == "pooled_complications":
                group_data = analyte_exp[analyte_exp[mode] == group]

            elif mode == "mutually_exclusive":
                group_data = analyte_exp[analyte_exp["Group"] == group]

            elif mode == "non_mutually_exclusive":
                group_data = analyte_exp[[group in SUBCONDITION_MAPPING[x] for x in analyte_exp["Subgroup"]]]

            
            if group_data.empty:
                continue
                
            # --- 1. Normality Testing (Maintained from your original code) ---
            #if len(group_data[m]) >= 3:
            #    statistic, p_value = stats.shapiro(group_data[m])
            #    normality[group] = {"statistic": statistic, "p-value": p_value}
            
            # --- 2. GEE Modeling ---
            # Define response (Y), predictors (X), and clusters (groups)
            Y = group_data[m]
            X = sm.add_constant(group_data["SampleGestAge"])
            subject_groups = group_data["SubjectID"]
            
            try:
                # We use sm.GEE (standard API rather than formula API) because 
                # analyte names often contain hyphens (e.g., IL-6) which break formulas.
                # Gaussian family + robust covariance is standard for unknown continuous distributions.
                gee_model = sm.GEE(
                    Y, X, 
                    groups=subject_groups,
                    cov_struct=sm.cov_struct.Exchangeable(),
                    family=sm.families.Gaussian()
                )
                result = gee_model.fit()
                
                # --- 3. Generate Predictions & Confidence Intervals ---
                # Create a smooth line of Gestational Ages for plotting
                x_min = group_data["SampleGestAge"].min()
                x_max = group_data["SampleGestAge"].max()
                x_pred = np.linspace(x_min, x_max, 100)
                
                # Design matrix for predictions
                X_pred = sm.add_constant(x_pred)
                
                # Mean predictions
                y_pred = result.predict(X_pred)
                
                # Manual standard error calculation for GEE predictions:
                # var(pred) = diag(X_pred * cov_params * X_pred^T)
                cov_matrix = result.cov_params().values
                
                # Highly efficient matrix calculation for the diagonal elements
                var_pred = np.sum((X_pred @ cov_matrix) * X_pred, axis=1)
                se_pred = np.sqrt(var_pred)
                
                # Upper and lower bounds
                y_lower = y_pred - (z_score * se_pred)
                y_upper = y_pred + (z_score * se_pred)
                
                # --- 4. Plotting ---
                color = colors.get(group, "black")
                plt.plot(x_pred, y_pred, label=group, color=color, linewidth=2)
                plt.fill_between(x_pred, y_lower, y_upper, color=color, alpha=0.2)
                
            except Exception as e:
                print(f"Warning: Could not fit GEE for {group} in {m}. Error: {e}")

        # Save normality stats
        temp_norm = pd.DataFrame(normality)
        try:
            temp_norm.to_csv(f"{dir_output}/shapiro-wilk-test/{tissue}_normality_tests_{m}_{mode}.csv")
        except:
            print(f"Issue with {m} file pathing.")
            continue
        
        # Format and save the plot
        plt.title(f"{modality} {m} in {tissue} Trajectory Plot - {mode}")
        plt.xlabel("Gestational Week")
        plt.ylabel(f"{m} log2 Normalized Protein Expression")
        
        legend = plt.legend(title="Group", loc="lower right")
        plt.setp(legend.get_texts(), fontsize='10')
        
        plt.savefig(f"{dir_output}/{tissue}_trajectory_plot_{m}_{mode}.jpeg", bbox_inches='tight')
        plt.close()

In [34]:
generate_trajectory_plots(list(elevated["analyte_ID"]), allPlasma, dir_output+"/elevated/pooled_complications", "plasma",  "LIPD", "pooled_complications")
generate_trajectory_plots(list(elevated["analyte_ID"]), allPlasma, dir_output+"/elevated/mutually_exclusive", "plasma",  "LIPD", "mutually_exclusive")
generate_trajectory_plots(list(elevated["analyte_ID"]), allPlasma, dir_output+"/elevated/non_mutually_exclusive", "plasma",  "LIPD", "non_mutually_exclusive")


In [36]:
generate_trajectory_plots(list(decreased["analyte_ID"]), allPlasma, dir_output+"/decreased/pooled_complications", "plasma",  "LIPD", "pooled_complications")
generate_trajectory_plots(list(decreased["analyte_ID"]), allPlasma, dir_output+"/decreased/mutually_exclusive", "plasma",  "LIPD", "mutually_exclusive")
generate_trajectory_plots(list(decreased["analyte_ID"]), allPlasma, dir_output+"/decreased/non_mutually_exclusive", "plasma",  "LIPD", "non_mutually_exclusive")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/genmod/generalized_estimating_equations.py:1300: IterationLimitWarning: Iteration limit reached prior to convergence
  warnings.warn("Iteration limit reached prior to convergence",
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/genmod/generalized_estimating_equations.py:1300: IterationLimitWarning: Iteration limit reached prior to convergence
  warnings.warn("Iteration limit reached prior to convergence",
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/genmod/generalized_estimating_equations.py:1300: IterationLimitWarning: Iteration limit reached prior to convergence
  warnings.warn("Iteration limit reached prior to convergence",
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/genmod/generalized_estimating_equations.py:1300: IterationLimitWarning: Iteration limit r

In [33]:
decreased

,Unnamed: 0,analyte_ID,total,complication_specific_1,most_extreme_2,most_prevalent,early_warning,most_extreme_1,most_extreme_4,most_extreme_5,most_persistent,complication_specific_4,complication_specific_3,most_extreme_3,complication_specific_2,complication_specific_5
0,11,DG(16:0_16:0)_POS__2,3,NaN,NaN,"{'n_patients_affected': 20, 'percent_complicat...","{'n_patients_decreased_at_earliest': 13, 'perc...","{'n_patients_affected': 10, 'median_MAD_score'...","{'n_patients_affected': 7, 'median_MAD_score':...","{'n_patients_affected': 8, 'median_MAD_score':...",NaN,NaN,NaN,NaN,NaN,NaN
1,29,TG(4:0_15:0_18:2)_POS,3,NaN,NaN,"{'n_patients_affected': 17, 'percent_complicat...","{'n_patients_decreased_at_earliest': 17, 'perc...",NaN,NaN,NaN,"{'n_patients_affected': 17, 'mean_outlier_time...",NaN,NaN,NaN,NaN,NaN
2,36,MG(27:2)_POS,3,NaN,NaN,"{'n_patients_affected': 26, 'percent_complicat...","{'n_patients_decreased_at_earliest': 24, 'perc...",NaN,NaN,NaN,NaN,NaN,"{'group': 'FGR', 'reference': 'sPTB', 'gestati...",NaN,NaN,NaN
3,51,PC(19:0COOH_15:0)_POS,3,NaN,"{'n_patients_affected': 16, 'median_MAD_score'...","{'n_patients_affected': 19, 'percent_complicat...","{'n_patients_decreased_at_earliest': 14, 'perc...","{'n_patients_affected': 12, 'median_MAD_score'...","{'n_patients_affected': 6, 'median_MAD_score':...","{'n_patients_affected': 11, 'median_MAD_score'...",NaN,NaN,NaN,"{'n_patients_affected': 15, 'median_MAD_score'...",NaN,NaN
4,79,TG(18:1_18:2_18:2)_POS,3,NaN,"{'n_patients_affected': 18, 'median_MAD_score'...","{'n_patients_affected': 22, 'percent_complicat...","{'n_patients_decreased_at_earliest': 15, 'perc...",NaN,NaN,"{'n_patients_affected': 12, 'median_MAD_score'...",NaN,NaN,NaN,"{'n_patients_affected': 19, 'median_MAD_score'...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1628,PC(P_POS__17,3,NaN,"{'n_patients_affected': 14, 'median_MAD_score'...","{'n_patients_affected': 18, 'percent_complicat...","{'n_patients_decreased_at_earliest': 14, 'perc...","{'n_patients_affected': 9, 'median_MAD_score':...","{'n_patients_affected': 5, 'median_MAD_score':...",NaN,NaN,NaN,NaN,"{'n_patients_affected': 14, 'median_MAD_score'...",NaN,NaN
60,1651,TG(10:1_18:2_18:3)_POS,4,NaN,NaN,"{'n_patients_affected': 20, 'percent_complicat...","{'n_patients_decreased_at_earliest': 15, 'perc...",NaN,NaN,"{'n_patients_affected': 11, 'median_MAD_score'...",NaN,NaN,NaN,"{'n_patients_affected': 18, 'median_MAD_score'...","{'group': 'FGR', 'reference': 'sPTB', 'gestati...",NaN
61,1697,TG(9:0_16:0_18:2)_POS__2,3,NaN,NaN,"{'n_patients_affected': 20, 'percent_complicat...","{'n_patients_decreased_at_earliest': 15, 'perc...",NaN,NaN,NaN,NaN,"{'group': 'FGR', 'reference': 'sPTB', 'gestati...",NaN,NaN,NaN,NaN
62,1725,LPC(20:3)_NEG,3,NaN,NaN,"{'n_patients_affected': 23, 'percent_complicat...","{'n_patients_decreased_at_earliest': 20, 'perc...",NaN,NaN,NaN,NaN,"{'group': 'HDP', 'reference': 'sPTB', 'gestati...",NaN,NaN,NaN,NaN
